# ⏩ Microsoft Foundry를 활용한 순차적 에이전트 워크플로우 (Python)

## 📋 고급 순차 처리 튜토리얼

이 노트북은 Microsoft Agent Framework를 사용한 <strong>순차적 워크플로우 패턴</strong>을 보여줍니다. 에이전트들이 특정 순서대로 실행되며 단계별로 데이터와 컨텍스트를 전달하는 정교한 다중 단계 처리 파이프라인을 구축하는 방법을 배우게 됩니다.

> **이전 참고 사항:** 이 샘플은 이전에 GitHub Models를 참조했습니다. GitHub Models는 2026년 7월에 폐지될 예정이므로, 이제는 <strong>Microsoft Foundry</strong>를 `FoundryChatClient`를 통해 사용하며, 이는 Azure OpenAI의 <strong>Responses API</strong>를 대상으로 합니다.

## 🎯 학습 목표

### 🔄 **순차 처리 패턴**
- **선형 워크플로우 설계**: 단계별 처리 파이프라인 생성
- **데이터 흐름 관리**: 순차적 에이전트 간 정보 전달
- **단계-승인 처리**: 체크포인트 및 검증 단계 구현
- **진행 상황 추적**: 워크플로우 실행 및 중간 결과 모니터링

### 🏗️ **기업용 파이프라인 아키텍처**
- **비즈니스 프로세스 모델링**: 실제 비즈니스 프로세스를 에이전트 워크플로우에 매핑
- **품질 보증**: 다단계 검증 및 검토 프로세스
- **문서 처리**: 순차적 문서 분석 및 변환
- **콘텐츠 제작**: 검토 및 승인 단계가 포함된 편집 워크플로우

### 📊 **고급 워크플로우 기능**
- **컨텍스트 유지**: 워크플로우 단계 간 상태 유지
- **오류 전파**: 순차 처리에서의 실패 처리
- **성능 최적화**: 효율적인 순차 실행 패턴
- **감사 추적**: 순차 작업의 완전한 추적

## ⚙️ 사전 요구 사항 및 설정

### 📦 <strong>종속성</strong>
```bash
pip install agent-framework -U
```

### 🔑 <strong>구성</strong>

Azure CLI(`az login`)에 로그인하여 `AzureCliCredential`이 인증할 수 있도록 한 후 Microsoft Foundry 프로젝트 세부 정보를 설정합니다.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

## 🏢 **기업용 순차 워크플로우 사용 사례**

### 📝 **문서 처리 파이프라인**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **품질 보증 워크플로우**
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **콘텐츠 제작 파이프라인**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **비즈니스 프로세스 자동화**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **순차 워크플로우 설계 원칙**

- **🔗 선형 진행**: 각 단계는 이전 단계의 출력에 의존
- **📋 상태 관리**: 모든 단계에서 컨텍스트와 데이터 유지
- **🛡️ 오류 처리**: 어떤 단계에서든 우아한 실패 관리
- **📊 진행 모니터링**: 각 단계 완료 및 성능 추적
- **🔄 단계 재사용성**: 재사용 가능한 워크플로우 컴포넌트 설계

정교한 순차 처리 워크플로우를 만들어 봅시다! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal message with an image of a
# living room. To keep the lesson focused on sequential workflow mechanics, this
# migration passes a textual description of the same scene as the workflow input.
# Agents accept a plain string, matching the basic and concurrent samples.
message = (
    "I am furnishing a modern living room and want pieces that fit a warm, "
    "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
    "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
    "Please find appropriate furniture and give the corresponding price for "
    "each piece, then produce a final purchase quote."
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
